In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import io
import matplotlib.patches as mpatches
from google.colab import files

# 1. Upload dos arquivos (selecione mqttsn.csv e p4ssn.csv)
print("Selecione os arquivos CSV gerados (mqttsn.csv e p4ssn.csv):")
uploaded = files.upload()

# 2. Carregamento e Tratamento
dfs = []
for filename in uploaded.keys():
    df_temp = pd.read_csv(io.BytesIO(uploaded[filename]))
    # Padroniza nomes de cenários para exibição
    df_temp['cenario'] = df_temp['cenario'].replace({'mqttsn': 'MQTT-SN', 'p4ssn': 'P4SSN'}).str.upper()
    dfs.append(df_temp)

df_completo = pd.concat(dfs, ignore_index=True)

def format_qos(row):
    # O valor 3 no CSV representa o QoS -1
    if row['qos'] == 3:
        return "QoS -1"

    label = f"QoS {int(row['qos'])}"
    if row['qos'] == 0:
        label += " (Com Retain)" if row['retain'] == 1 else " (Sem Retain)"
    return label

df_completo['QoS_Config'] = df_completo.apply(format_qos, axis=1)

# Médias para o gráfico
df_plot = df_completo.groupby(['cenario', 'QoS_Config'])['t_flow_ms'].mean().reset_index()
df_plot.rename(columns={'cenario': 'Topologia', 't_flow_ms': 'Tempo Médio do Fluxo (ms)'}, inplace=True)

# 3. Configuração Visual
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 7))

# Ordem do eixo X
ordem_x = ["QoS -1", "QoS 0 (Sem Retain)", "QoS 0 (Com Retain)", "QoS 1", "QoS 2"]
palette_colors = sns.color_palette("viridis", 2)

ax = sns.barplot(
    data=df_plot,
    x='QoS_Config',
    y='Tempo Médio do Fluxo (ms)',
    hue='Topologia',
    palette=palette_colors,
    order=ordem_x
)

# 4. Aplicação de Hachuras e Legenda Customizada
patterns = ['...', '///']
topologias = sorted(df_plot['Topologia'].unique())

for i, patch in enumerate(ax.patches):
    if i < len(ordem_x):
        patch.set_hatch(patterns[0])
    else:
        patch.set_hatch(patterns[1])

# Legenda customizada com hachuras
patch1 = mpatches.Patch(facecolor=palette_colors[0], hatch=patterns[0], label=topologias[0])
patch2 = mpatches.Patch(facecolor=palette_colors[1], hatch=patterns[1], label=topologias[1])

plt.legend(handles=[patch1, patch2], title='Protocolo', loc='upper left', frameon=True)

# Finalização
plt.title('Compatibilidade Funcional: MQTT-SN x P4SSN', fontsize=15, pad=20)
plt.xlabel('Nível de QoS', fontsize=12)
plt.ylabel('Tempo Médio do Fluxo de Publicação (ms)', fontsize=12)

# Nota: O loop de ax.annotate que adicionava os valores sobre as barras foi removido.

plt.tight_layout()
plt.savefig("grafico_fluxo_mqttsn_p4ssn.eps", format='eps', bbox_inches='tight')
plt.show()